# Polynomial Factorization
## Division, gcds, roots, and what “irreducible” means over ℚ and 𝔽ₚ

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/eduenez/courses/blob/main/2026fall/mat5173/downloads/public/activities/polynomial-factorization.ipynb)
[![License: CC BY 4.0](https://img.shields.io/badge/License-CC%20BY%204.0-lightgrey.svg)](https://creativecommons.org/licenses/by/4.0/)

---

In $F[x]$, with $F$ a field, polynomials behave remarkably like integers:
there is a division algorithm, a Euclidean algorithm, a Bézout identity, and
unique factorization into irreducibles. This notebook builds all of that from
scratch, in a few dozen lines of plain Python, and then uses it to watch one
integer polynomial factor in completely different ways depending on which field
its coefficients are read in.

1. **Division with remainder** — over a field, and over $\mathbb{Z}$ when the
   divisor is monic (Ash 2.5.1).
2. **The Euclidean algorithm in $F[x]$** — gcds and Bézout coefficients, so
   that $(f, g) = (d)$.
3. **Roots** — the remainder theorem, and "at most $n$ roots" in a field,
   tested experimentally over $\mathbb{F}_p$.
4. **Irreducible depends on the field** — factoring $x^2+1$ and $x^4+1$ over
   $\mathbb{Q}$ and over many $\mathbb{F}_p$.
5. **Unique factorization, up to units and order** — what the "up to" means
   in $F[x]$.
6. **Looking ahead to §2.9** — reduction mod $p$, and Eisenstein's criterion.

Nothing here has a grade attached and there's nothing to submit. Tinker
freely — *Runtime → Run all* always resets you to a clean state.



---
# 0. Setup

A polynomial is a list of coefficients, **constant term first**:
`[1, 0, 3]` means $1 + 3x^2$. The functions below take the field as an
argument, so the *same* code runs over $\mathbb{Q}$ (exact fractions, no
rounding) and over $\mathbb{F}_p$ (integers mod $p$). All that a field has to
supply is a way to turn an integer into one of its elements and a way to
invert a nonzero element.


In [1]:
from fractions import Fraction
from itertools import product

class Field:
    def __init__(self, name, elt, inv, elements=None):
        self.name, self.elt, self.inv = name, elt, inv
        self.elements = elements        # a list, when the field is finite

QQ = Field("Q", Fraction, lambda a: 1 / a)

def GF(p):
    return Field(f"F_{p}", lambda a: a % p, lambda a: pow(a, -1, p), list(range(p)))

def poly(coeffs, K):
    # Read a list of integers (or fractions) as a polynomial over K.
    return trim([K.elt(c) for c in coeffs])

def trim(f):
    f = list(f)
    while f and f[-1] == 0:
        f.pop()
    return f                            # the zero polynomial is []

def deg(f):
    return len(f) - 1 if f else float("-inf")   # Ash's convention: deg 0 = -infinity

def add(f, g, K):
    n = max(len(f), len(g))
    f, g = f + [0] * (n - len(f)), g + [0] * (n - len(g))
    return trim([K.elt(a + b) for a, b in zip(f, g)])

def neg(f, K):
    return [K.elt(-a) for a in f]

def mul(f, g, K):
    if not f or not g:
        return []
    h = [0] * (len(f) + len(g) - 1)
    for i, a in enumerate(f):
        for j, b in enumerate(g):
            h[i + j] += a * b
    return trim([K.elt(c) for c in h])

def monic(f, K):
    # Divide by the leading coefficient -- a unit, since K is a field.
    u = K.inv(f[-1])
    return [K.elt(u * a) for a in f]

def show(f):
    if not f:
        return "0"
    out = ""
    for i in range(len(f) - 1, -1, -1):
        c = f[i]
        if c == 0:
            continue
        sign, c = ("-", -c) if c < 0 else ("+", c)
        x = "" if i == 0 else ("x" if i == 1 else f"x^{i}")
        cs = str(c)
        if x and c == 1:
            term = x
        elif x:
            term = (f"({cs})" if "/" in cs else cs) + "*" + x
        else:
            term = cs
        out += (f" {sign} " if out else ("-" if sign == "-" else "")) + term
    return out

print(show(poly([1, 0, 3], QQ)), "   over Q")
print(show(poly([1, 0, 3], GF(3))), "          over F_3  (3x^2 is zero there)")


3*x^2 + 1    over Q
1           over F_3  (3x^2 is zero there)


---
# 1. Division with remainder

Ash 2.5.1: if $g$ is **monic**, then for every $f$ there are unique $q, r$ with
$f = qg + r$ and $\deg r < \deg g$ — over *any* commutative ring. Over a
**field**, $g$ can be any nonzero polynomial, because we can always divide by
its leading coefficient. The code is ordinary long division, and the one
line that needs a field is the one computing `lead`.


In [2]:
def divmod_poly(f, g, K):
    if not g:
        raise ZeroDivisionError("division by the zero polynomial")
    q, r = [], list(f)
    lead = K.inv(g[-1])                 # <-- the only place a field is needed
    while r and deg(r) >= deg(g):
        shift = deg(r) - deg(g)
        c = K.elt(r[-1] * lead)
        term = [0] * shift + [c]        # c * x^shift
        q = add(q, term, K)
        r = add(r, neg(mul(term, g, K), K), K)
    return q, r

# Ash's own worked example (§2.5): 2x^3 + 10x^2 + 16x + 10 divided by x^2 + 3x + 5.
f, g = poly([10, 16, 10, 2], QQ), poly([5, 3, 1], QQ)
q, r = divmod_poly(f, g, QQ)
print("q =", show(q), "   r =", show(r))
assert add(mul(q, g, QQ), r, QQ) == f


q = 2*x + 4    r = -6*x - 10


Ash does that example over $\mathbb{Z}$, and the answer, $q = 2x+4$ and
$r = -6x-10$, has integer coefficients even though we ran it over
$\mathbb{Q}$. That is no accident: with a **monic** divisor, `lead` is $1$, so
long division never leaves the ring the coefficients started in.

Now try a divisor that is *not* monic:


In [3]:
f, g = poly([1, 0, 0, 1], QQ), poly([1, 2], QQ)      # x^3 + 1  divided by  2x + 1
q, r = divmod_poly(f, g, QQ)
print("q =", show(q), "   r =", show(r))


q = (1/2)*x^2 - (1/4)*x + 1/8    r = 7/8


**Try it:** the quotient has non-integer coefficients. Is there *any* pair
$q, r \in \mathbb{Z}[x]$ with $x^3 + 1 = q\,(2x+1) + r$ and $\deg r < 1$?
(Compare leading coefficients.) This is exactly why Ash asks for $g$ monic
when $R$ is not a field.


---
# 2. The Euclidean algorithm in $F[x]$

Because $F[x]$ has a division algorithm, Euclid's algorithm runs there
exactly as it does in $\mathbb{Z}$ (Ash's §2.5 problems 1–6 walk through it
for integers). Keeping track of the quotients also produces the **Bézout
coefficients**: polynomials $a, b$ with $a f + b g = d$, where
$d = \gcd(f, g)$ is taken monic. This is the computational content of the
statement that the ideal $(f, g)$ is the principal ideal $(d)$.


In [4]:
def ext_gcd(f, g, K):
    # Returns (d, a, b) with a*f + b*g = d, and d = gcd(f, g) monic.
    r0, r1 = f, g
    a0, a1 = poly([1], K), []
    b0, b1 = [], poly([1], K)
    while r1:
        q, r = divmod_poly(r0, r1, K)
        r0, r1 = r1, r
        a0, a1 = a1, add(a0, neg(mul(q, a1, K), K), K)
        b0, b1 = b1, add(b0, neg(mul(q, b1, K), K), K)
    u = K.inv(r0[-1])                   # normalize: the gcd is only defined up to a unit
    scale = lambda h: [K.elt(u * c) for c in h]
    return scale(r0), scale(a0), scale(b0)

f = poly([-1, 0, 0, 0, 1], QQ)          # x^4 - 1
g = poly([-1, 0, 0, 0, 0, 0, 1], QQ)    # x^6 - 1
d, a, b = ext_gcd(f, g, QQ)
print("gcd =", show(d))
print("a   =", show(a))
print("b   =", show(b))
assert add(mul(a, f, QQ), mul(b, g, QQ), QQ) == d


gcd = x^2 - 1
a   = -x^2
b   = 1


**Try it:** $x^4 - 1$ and $x^6 - 1$ have gcd $x^{\gcd(4,6)} - 1$. Change the
exponents to other pairs $m, n$ and check that $\gcd(x^m - 1,\, x^n - 1) =
x^{\gcd(m,n)} - 1$ every time. Then run the same pair over `GF(2)` or
`GF(3)`: does the answer change?


---
# 3. Roots

The **remainder theorem** (Ash 2.5.2) says that dividing $f$ by $x - a$ leaves
the constant remainder $f(a)$. So $a$ is a root exactly when $x - a$ divides
$f$. Peeling off one linear factor per root is what shows that a nonzero
polynomial of degree $n$ over an integral domain has at most $n$ roots
(Ash 2.5.3).


In [5]:
def evaluate(f, a, K):
    v = K.elt(0)
    for c in reversed(f):               # Horner's rule
        v = K.elt(v * a + c)
    return v

K = QQ
f = poly([-6, 11, -6, 1], K)            # (x - 1)(x - 2)(x - 3)
for a in [1, 2, 3, 4]:
    q, r = divmod_poly(f, poly([-a, 1], K), K)
    print(f"a = {a}:  remainder {show(r):>3}   f(a) = {evaluate(f, K.elt(a), K)}")


a = 1:  remainder   0   f(a) = 0
a = 2:  remainder   0   f(a) = 0
a = 3:  remainder   0   f(a) = 0
a = 4:  remainder   6   f(a) = 6


In a **finite** field we can simply try every element. Here is an experiment:
for a prime $p$, count the roots of $x^d - 1$ in $\mathbb{F}_p$ for each
$d = 1, \dots, 12$.


In [6]:
def roots(f, K):
    return [a for a in K.elements if evaluate(f, a, K) == 0]

p = 13
K = GF(p)
for d in range(1, 13):
    f = poly([-1] + [0] * (d - 1) + [1], K)          # x^d - 1
    label = "x - 1" if d == 1 else f"x^{d} - 1"
    print(f"{label:>8} has {len(roots(f, K)):2d} root(s) in F_{p}")


   x - 1 has  1 root(s) in F_13
 x^2 - 1 has  2 root(s) in F_13
 x^3 - 1 has  3 root(s) in F_13
 x^4 - 1 has  4 root(s) in F_13
 x^5 - 1 has  1 root(s) in F_13
 x^6 - 1 has  6 root(s) in F_13
 x^7 - 1 has  1 root(s) in F_13
 x^8 - 1 has  4 root(s) in F_13
 x^9 - 1 has  3 root(s) in F_13
x^10 - 1 has  2 root(s) in F_13
x^11 - 1 has  1 root(s) in F_13
x^12 - 1 has 12 root(s) in F_13


Theorem 2.5.3 promises at most $d$ roots, and for $p = 13$ some rows reach
that bound exactly while others fall well short.

**Try it:**
1. Which $d$ give exactly $d$ roots? Change `p` to $7$, $11$, $17$ and form a
   conjecture in terms of $d$ and $p - 1$. (The roots of $x^d - 1$ are the
   elements of order dividing $d$ in the group $\mathbb{F}_p^\times$. What
   do you know about that group's order?)
2. For the $d$ that fall short, how many roots are there? Guess a formula in
   terms of $\gcd(d, p-1)$.
3. The polynomial $x^p - x$ has degree $p$. How many roots does it have in
   $\mathbb{F}_p$? Is it the zero polynomial? (Compare Ash §2.5, problems
   9–10.)


---
# 4. Irreducible depends on the field

Over a field $F$, the units of $F[x]$ are the nonzero constants, and a
nonconstant polynomial is **irreducible** if it is not a product of two
polynomials of lower degree. In $\mathbb{F}_p[x]$ we can factor by plain trial
division: remove the smallest-degree monic divisor, then repeat. It suffices
to try divisors of degree at most $\deg f / 2$, and the first divisor found
is automatically irreducible, much as the smallest factor $> 1$ of an integer
is automatically prime.


In [7]:
def monic_polys(n, K):
    for tail in product(K.elements, repeat=n):
        yield list(tail) + [K.elt(1)]

def factor(f, K):
    # Returns (unit, [monic irreducible factors, with repetition]); K must be finite.
    unit, f = f[-1], monic(f, K)
    factors, n = [], 1
    while deg(f) >= 2 * n:
        for g in monic_polys(n, K):
            q, r = divmod_poly(f, g, K)
            if not r:
                factors.append(g)
                f = q
                break
        else:
            n += 1                      # nothing of degree n divides f; try n + 1
    if deg(f) > 0:
        factors.append(f)               # what remains has no smaller factor: irreducible
    return unit, factors

def show_factored(unit, factors):
    body = " * ".join(f"({show(g)})" for g in factors) or "1"
    return body if unit == 1 else f"{unit} * {body}"

for p in [2, 3, 5, 7, 11, 13, 17]:
    K = GF(p)
    print(f"over F_{p:<2}:  x^2 + 1 = {show_factored(*factor(poly([1, 0, 1], K), K))}")


over F_2 :  x^2 + 1 = (x + 1) * (x + 1)
over F_3 :  x^2 + 1 = (x^2 + 1)
over F_5 :  x^2 + 1 = (x + 2) * (x + 3)
over F_7 :  x^2 + 1 = (x^2 + 1)
over F_11:  x^2 + 1 = (x^2 + 1)
over F_13:  x^2 + 1 = (x + 5) * (x + 8)
over F_17:  x^2 + 1 = (x + 4) * (x + 13)


Over $\mathbb{Q}$ (or even $\mathbb{R}$), $x^2 + 1$ is irreducible: a
factorization of a quadratic would need a linear factor, hence a root, and
$a^2 + 1 = 0$ has no rational solution. Over $\mathbb{F}_p$ it splits for some
primes and stays irreducible for others.

**Try it:** extend the list of primes up to $100$ and sort them into "splits"
and "irreducible". The dividing line depends on $p \bmod 4$. Can you explain
it using what Section 3 suggested about $\mathbb{F}_p^\times$? (A root of
$x^2 + 1$ is an element of order $4$.)


Now a more dramatic example. The polynomial $x^4 + 1$ has **no factorization
at all** over $\mathbb{Q}$. It has no rational roots (the only candidates are
$\pm 1$), so a factorization would have to be into two quadratics. Comparing
coefficients rules that out; see Exercise 4 below. Yet modulo every prime we
try, it breaks apart:


In [8]:
for p in [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31]:
    K = GF(p)
    f = poly([1, 0, 0, 0, 1], K)
    unit, fs = factor(f, K)
    degrees = sorted(deg(g) for g in fs)
    print(f"over F_{p:<2}:  degrees {degrees}   {show_factored(unit, fs)}")


over F_2 :  degrees [1, 1, 1, 1]   (x + 1) * (x + 1) * (x + 1) * (x + 1)
over F_3 :  degrees [2, 2]   (x^2 + x + 2) * (x^2 + 2*x + 2)
over F_5 :  degrees [2, 2]   (x^2 + 2) * (x^2 + 3)
over F_7 :  degrees [2, 2]   (x^2 + 3*x + 1) * (x^2 + 4*x + 1)
over F_11:  degrees [2, 2]   (x^2 + 3*x + 10) * (x^2 + 8*x + 10)
over F_13:  degrees [2, 2]   (x^2 + 5) * (x^2 + 8)
over F_17:  degrees [1, 1, 1, 1]   (x + 2) * (x + 8) * (x + 9) * (x + 15)
over F_19:  degrees [2, 2]   (x^2 + 6*x + 18) * (x^2 + 13*x + 18)
over F_23:  degrees [2, 2]   (x^2 + 5*x + 1) * (x^2 + 18*x + 1)
over F_29:  degrees [2, 2]   (x^2 + 12) * (x^2 + 17)
over F_31:  degrees [2, 2]   (x^2 + 8*x + 1) * (x^2 + 23*x + 1)


Every prime gives factors of degree at most $2$. So "irreducible" is not a
property of a list of integer coefficients; it is a property of a polynomial
**in a particular ring**. That is the same lesson as "unit", which depends on
the ring and not on the element alone.

**Try it:** can you find a prime $p$ for which $x^4 + 1$ stays irreducible
over $\mathbb{F}_p$? Search as far as you have patience for. (It is a theorem
that you will not find one; Section 6 explains why that matters.)


---
# 5. Unique factorization, up to units and order

$F[x]$ is a UFD (Ash §2.6). "Unique" there always means **up to order and up
to associates**, and in $F[x]$ two polynomials are associates exactly when one
is a nonzero-constant multiple of the other. Insisting that every
irreducible factor be **monic** picks one representative from each class of
associates, so that the factorization becomes literally unique: one leading
unit, and a multiset of monic irreducibles.

Below, the same polynomial over $\mathbb{F}_7$ is built from two sets of
factors that *look* different. The monic normalization shows they are the
same factorization.


In [9]:
K = GF(7)
A = [poly([2, 4], K), poly([3, 0, 6], K), poly([1, 1], K)]     # (4x+2)(6x^2+3)(x+1)
B = [poly([5, 5], K), poly([1, 0, 2], K), poly([4, 1], K)]     # (5x+5)(2x^2+1)(x+4)

prod_A, prod_B = poly([1], K), poly([1], K)
for g in A: prod_A = mul(prod_A, g, K)
for g in B: prod_B = mul(prod_B, g, K)
print("product A:", show(prod_A))
print("product B:", show(prod_B))
print("same polynomial:", prod_A == prod_B)
print()
print("monic factors of A:", sorted(show(monic(g, K)) for g in A))
print("monic factors of B:", sorted(show(monic(g, K)) for g in B))
print("factor() says:     ", show_factored(*factor(prod_A, K)))


product A: 3*x^4 + x^3 + 3*x^2 + 4*x + 6
product B: 3*x^4 + x^3 + 3*x^2 + 4*x + 6
same polynomial: True

monic factors of A: ['x + 1', 'x + 4', 'x^2 + 4']
monic factors of B: ['x + 1', 'x + 4', 'x^2 + 4']
factor() says:      3 * (x + 1) * (x + 4) * (x^2 + 4)


**Try it:** is $x^2 + 4$ irreducible over $\mathbb{F}_7$? Check with
`roots(...)`, and explain why a root check settles the question for
quadratics and cubics but **not** for quartics. (Look back at $x^4 + 1$.)


---
# 6. Looking ahead to §2.9 (after Midterm 1)

*Nothing in this section is needed yet. It previews Ash §2.9, where it will
be proved.*

**Reduction mod $p$.** Gauss's lemma (Ash §2.9) implies that a monic integer
polynomial which factors over $\mathbb{Q}$ already factors over $\mathbb{Z}$,
into monic factors. Reducing that factorization mod $p$ then gives a
factorization over $\mathbb{F}_p$ with the same degrees. Read backwards:

> if a monic $f \in \mathbb{Z}[x]$ is irreducible over **some** $\mathbb{F}_p$,
> then $f$ is irreducible over $\mathbb{Q}$.

So a single prime can *prove* irreducibility over $\mathbb{Q}$. Section 4
shows the converse fails: $x^4 + 1$ is irreducible over $\mathbb{Q}$, yet no
prime certifies it.


In [10]:
coeffs = [1, 1, 0, 0, 1]                                        # x^4 + x + 1
for p in [2, 3, 5]:
    K = GF(p)
    unit, fs = factor(poly(coeffs, K), K)
    verdict = "irreducible" if len(fs) == 1 else "reducible"
    print(f"over F_{p}:  {show_factored(unit, fs):<34} {verdict}")


over F_2:  (x^4 + x + 1)                      irreducible
over F_3:  (x + 2) * (x^3 + x^2 + x + 2)      reducible
over F_5:  (x + 2) * (x^3 + 3*x^2 + 4*x + 3)  reducible


It factors mod $3$ and mod $5$, but it is irreducible mod $2$, and that one
prime is enough: $x^4 + x + 1$ is irreducible over $\mathbb{Q}$.

**Eisenstein's criterion** (Ash 2.9.7) gives irreducibility without any
search. If a prime $p$ divides every coefficient except the leading one, and
$p^2$ does not divide the constant term, then the polynomial is irreducible
over $\mathbb{Q}$. It does not apply directly to
$\Phi_5(x) = x^4 + x^3 + x^2 + x + 1$, but it does apply after the
substitution $x \mapsto x + 1$:


In [11]:
from math import comb

def shift_by_one(coeffs):
    # Integer coefficients of f(x + 1), constant term first.
    n = len(coeffs) - 1
    return [sum(coeffs[k] * comb(k, j) for k in range(j, n + 1)) for j in range(n + 1)]

def eisenstein(coeffs, p):
    *lower, lead = coeffs
    return lead % p != 0 and all(c % p == 0 for c in lower) and coeffs[0] % (p * p) != 0

phi5 = [1, 1, 1, 1, 1]
shifted = shift_by_one(phi5)
print("Phi_5(x)     coefficients:", phi5, "  Eisenstein at 5?", eisenstein(phi5, 5))
print("Phi_5(x + 1) coefficients:", shifted, "  Eisenstein at 5?", eisenstein(shifted, 5))


Phi_5(x)     coefficients: [1, 1, 1, 1, 1]   Eisenstein at 5? False
Phi_5(x + 1) coefficients: [5, 10, 10, 5, 1]   Eisenstein at 5? True


A factorization of $\Phi_5(x)$ would give one of $\Phi_5(x+1)$ with the same
degrees, so $\Phi_5$ is irreducible over $\mathbb{Q}$. **Try it:** repeat with
$\Phi_7(x) = x^6 + \dots + x + 1$ and $p = 7$. Then try the same trick on
$x^5 + x^4 + x^3 + x^2 + x + 1 = (x^6 - 1)/(x - 1)$ with $p = 3$: is Eisenstein's failure there
telling you something true? (Try `factor` over a few $\mathbb{F}_p$, or look
for a rational root.)


---
# 7. Summary

- Over a field, $F[x]$ has a division algorithm. Over any commutative ring,
  division by a **monic** polynomial still works.
- The Euclidean algorithm gives $\gcd(f, g)$ together with Bézout
  coefficients, so every ideal $(f, g)$ of $F[x]$ is principal.
- $a$ is a root of $f$ exactly when $(x - a) \mid f$, so a degree-$n$
  polynomial over a domain has at most $n$ roots. Over $\mathbb{F}_p$ the
  bound is attained or missed according to the structure of
  $\mathbb{F}_p^\times$.
- Irreducibility belongs to a polynomial **in a ring**. $x^2 + 1$ and
  $x^4 + 1$ are irreducible over $\mathbb{Q}$ but factor over many or all
  $\mathbb{F}_p$.
- Factorization in $F[x]$ is unique up to order and nonzero constants.
  Making every factor monic removes the ambiguity.


---
# 8. Exercises

1. **Bézout by hand.** Over $\mathbb{Q}$, find $\gcd(x^3 - 2x^2 + x - 2,\;
   x^2 + x - 6)$ and polynomials $a, b$ with $a f + b g = \gcd$, first on
   paper, then with `ext_gcd`. Is the answer different over $\mathbb{F}_5$?
2. **Lagrange interpolation** (Ash §2.5, problems 7–8). Write a function that,
   given distinct $a_0, \dots, a_n$ and arbitrary $b_0, \dots, b_n$ in a field,
   returns the unique polynomial of degree $\le n$ with $f(a_i) = b_i$. Test it
   over `QQ` and over `GF(11)`. What goes wrong if two of the $a_i$ coincide?
3. **When does $x^2 + x + 1$ split?** Repeat the $x^2 + 1$ experiment of
   Section 4 for $x^2 + x + 1$. The dividing line is now $p \bmod 3$. Why?
   (What is the multiplicative order of a root?)
4. **$x^4 + 1$ over $\mathbb{Q}$.** Suppose
   $x^4 + 1 = (x^2 + ax + b)(x^2 + cx + d)$ with $a, b, c, d \in \mathbb{Q}$.
   Compare coefficients and show that it is impossible. (Taking Gauss's lemma
   on trust, you may assume $a, b, c, d \in \mathbb{Z}$, which shortens the
   argument considerably.) Then, over $\mathbb{R}$, find the factorization
   that *does* exist.
5. **A polynomial that vanishes everywhere.** Over $\mathbb{F}_p$, factor
   $x^p - x$ with `factor` for a few primes. Explain the answer using
   Fermat's little theorem, and reconcile it with Theorem 2.5.3.


---
## License

This notebook is released under the [Creative Commons Attribution 4.0
International License (CC BY 4.0)](https://creativecommons.org/licenses/by/4.0/). You are free to copy, adapt,
and redistribute it, including for commercial purposes, provided you give
appropriate credit.

Suggested attribution: *"Polynomial Factorization" by [Eduardo Dueñez](https://supernumero.us/about),
licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).*
